# Interactive MMI Designer

Design and simulate Multi-Mode Interferometers (MMI) interactively.
Adjust parameters to see real-time intensity profiles and phasor contributions at the output.

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from helios.sim.mmi import compute_contributions
from helios.sim import simulate_contributions

In [2]:
# --- UI Layout Construction ---

style = {'description_width': 'initial'}

# 1. Global Parameters
N_slider = widgets.IntSlider(value=2, min=2, max=8, step=1, description='N Inputs:', style=style)
M_slider = widgets.IntSlider(value=2, min=2, max=8, step=1, description='M Outputs:', style=style)
L_input = widgets.FloatText(value=0, description='L (um) [0=Auto]:', style=style)
W_slider = widgets.FloatSlider(value=10.0, min=2.0, max=50.0, step=0.5, description='W (um):', style=style)
n_eff_slider = widgets.FloatSlider(value=2.0458, min=1.0, max=4.0, step=0.0001, description='n_eff:', style=style)
num_modes_slider = widgets.IntSlider(value=50, min=10, max=200, step=10, description='Num Modes:', style=style)

# Container for Input Amplitudes/Phases (Dynamic)
inputs_container = widgets.VBox([])

def build_input_controls(N):
    controls = []
    for i in range(N):
        amp = widgets.FloatSlider(value=1.0 if i==0 else 1.0, min=0.0, max=1.0, step=0.1, description=f'Amp {i+1}:', layout=widgets.Layout(width='95%'))
        phase = widgets.FloatSlider(value=0.0, min=0.0, max=2*np.pi, step=0.1, description=f'Phase {i+1} (rad):', layout=widgets.Layout(width='95%'))
        controls.append(widgets.HBox([amp, phase]))
    return controls

def update_input_container(change):
    N = change['new']
    children = build_input_controls(N)
    inputs_container.children = children

N_slider.observe(update_input_container, names='value')
# Initialize container
update_input_container({'new': N_slider.value})

# Plotting Function
output_plot = widgets.Output()

def update_plot(b=None):
    # Gather inputs
    N = N_slider.value
    M = M_slider.value
    L_val = L_input.value * 1e-6 # convert to m
    if L_val == 0:
        L_val = None
    W_val = W_slider.value * 1e-6
    n_eff_val = n_eff_slider.value
    n_modes_val = num_modes_slider.value
    
    # Construct complex input vector
    # inputs_container has VBox children which are HBoxes [amp, phase]
    amplitudes = []
    for hbox in inputs_container.children:
        amp = hbox.children[0].value
        phase = hbox.children[1].value
        amplitudes.append(amp * np.exp(1j * phase))
        
    # Normalize energy
    total_pow = sum(np.abs(a)**2 for a in amplitudes)
    if total_pow > 0:
        amplitudes = [a / np.sqrt(total_pow) for a in amplitudes]

    with output_plot:
        clear_output(wait=True)
        
        # Run Simulation with coarse Z resolution for speed, but ensure last frame is accurate
        # We actually only need the final state for the phasor plot, but we need the map for context.
        # Let's use a moderate resolution.
        try:
            data = compute_contributions(
                N=N, M=M, L=L_val, W=W_val, n_eff=n_eff_val,
                wavelength=1.55e-6, input_amplitudes=amplitudes,
                num_modes=n_modes_val,
                z_resolution=W_val*2, # Very coarse Z steps for speed, we mainly care about the End.
                 # Wait, if we use very coarse z_res, the map looks bad.
                 # Let's use roughly 50 steps.
                num_z_steps=50,
                verbose=False
            )
        except Exception as e:
            print(f"Simulation Error: {e}")
            return

        # Extract Data
        z_grid = data['z_grid']
        x_grid = data['x_grid']
        intensity_map = data['intensity_total_evol']
        phasors = data['phasors']
        input_pos = data['input_positions']
        output_pos = data['output_positions']
        L_sim = data['L']

        # Plotting
        fig = plt.figure(figsize=(12, 10))
        gs = fig.add_gridspec(3, M, height_ratios=[1.5, 1, 1.5])

        # 1. Intensity Map (Top)
        ax_map = fig.add_subplot(gs[0, :])
        extent = [0, L_sim*1e6, 0, W_val*1e6]
        im = ax_map.imshow(intensity_map.T, origin='lower', aspect='auto', extent=extent, cmap='inferno')
        ax_map.set_xlabel('z [um]')
        ax_map.set_ylabel('x [um]')
        ax_map.set_title(f'Intensity Map (L={L_sim*1e6:.1f} um)')
        
        # Markers
        ax_map.scatter([0]*N, [p*1e6 for p in input_pos], color='w', s=10)
        ax_map.scatter([L_sim*1e6]*M, [p*1e6 for p in output_pos], color='w', s=10)

        # 2. Cross Section at Output (Middle)
        ax_prof = fig.add_subplot(gs[1, :])
        ax_prof.plot(x_grid*1e6, intensity_map[-1, :], 'b-', lw=2)
        for p in output_pos:
            ax_prof.axvline(x=p*1e6, color='k', linestyle=':', alpha=0.5)
        ax_prof.set_xlim(0, W_val*1e6)
        ax_prof.set_xlabel('x [um]')
        ax_prof.set_ylabel('Intensity')
        ax_prof.set_title('Output Profile')

        # 3. Polar Plots (Bottom)
        colors = plt.cm.get_cmap('hsv', N+1)
        # Max coupling for scale
        max_val = np.max(np.abs(phasors[-1, :, :]))
        limit = max_val * 1.1 if max_val > 1e-6 else 1.0

        for j in range(M):
            ax_p = fig.add_subplot(gs[2, j], projection='polar')
            ax_p.set_title(f'Out {j+1}')
            ax_p.set_ylim(0, limit)
            
            # Contributions
            for i in range(N):
                val = phasors[-1, j, i]
                ax_p.plot([0, np.angle(val)], [0, np.abs(val)], color=colors(i), lw=2, label=f'In {i+1}')
            
            # Total
            tot = np.sum(phasors[-1, j, :])
            ax_p.plot([0, np.angle(tot)], [0, np.abs(tot)], 'k--', lw=2, label='Total')
            
            if j == M-1:
                ax_p.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=7)

        plt.tight_layout()
        plt.show()

# Button to Trigger Plot
run_btn = widgets.Button(
    description="Simulate",
    button_style='success',
    layout=widgets.Layout(width='100%')
)
run_btn.on_click(update_plot)

# Layout
ui = widgets.VBox([
    widgets.HBox([N_slider, M_slider]),
    widgets.HBox([L_input, W_slider]),
    widgets.HBox([n_eff_slider, num_modes_slider]),
    widgets.Label("Inputs Configuration:"),
    inputs_container,
    run_btn
])

display(ui, output_plot)

# Initial Run
update_plot()

Output()

In [3]:
simulate_contributions(
    N=2,
    M=2,
    L=100e-6,
    W=10.0e-6,
    n_eff=2.0458,
    wavelength=1.55e-6,
    input_amplitudes=np.sqrt(1/2)*np.array([1, 1j], dtype=complex),
    num_modes=50,
    z_resolution=1.0e-6, # Removed to use default high-res (lambda/30)
    output_file="2x2_nuller_mmi.mp4",
    verbose=True
)

Calculated num_z_steps = 102 for L = 100.0 um
Injecting input vector: [0.70710678+0.j         0.        +0.70710678j]


Simulating Propagation: 100%|██████████| 102/102 [00:00<00:00, 5823.12step/s]

Computing separate field contributions (Parallel)...


Generating contributions animation frames in parallel for 2x2_nuller_mmi.mp4...


Rendering Frames: 100%|██████████| 102/102 [00:23<00:00,  4.37it/s]


Stitching frames with ffmpeg...
Injecting input vector: [0.70710678+0.j         0.        +0.70710678j]


Simulating Propagation: 100%|██████████| 102/102 [00:00<00:00, 4363.41step/s]

Output amplitudes: [0.07768305+0.75896541j 0.22014726-0.22126525j]
Output amplitudes: [0.07768305+0.75896541j 0.22014726-0.22126525j]


array([0.07768305+0.75896541j, 0.22014726-0.22126525j])

In [4]:
simulate_contributions(
    N=4,
    M=4,
    L=400e-6,
    W=20e-6,
    n_eff=2.0458,
    wavelength=1.55e-6,
    input_amplitudes=np.sqrt(1/4)*np.array([1, 1j, 1, 1j], dtype=complex),
    num_modes=50,
    z_resolution=1.0e-6, # Removed to use default high-res (lambda/30)
    output_file="4x4_kernel_nuller_mmi.mp4", 
    verbose=True
)

Calculated num_z_steps = 402 for L = 400.0 um
Injecting input vector: [0.5+0.j  0. +0.5j 0.5+0.j  0. +0.5j]


Simulating Propagation: 100%|██████████| 402/402 [00:00<00:00, 4517.59step/s]

Computing separate field contributions (Parallel)...


Generating contributions animation frames in parallel for 4x4_kernel_nuller_mmi.mp4...


Rendering Frames:   4%|▍         | 16/402 [00:02<01:02,  6.18it/s]

KeyboardInterrupt: 